# SQL Join Examples (Mcor params exploration)

## Connect to DB

In [1]:
from odyn import Database

[INFO] Import Database and Group classes for db UI and help functions!
[INFO] You can use 'from odyn import Database, Group' to import them.
[INFO] Hovering over 'Database' and 'Group' will give you some tips.


In [2]:
db = Database("tmp/test_server")

[INFO] Connected to the database at: '/Users/vinicius/GitHub/Projects/ODyn/tmp/test_server/.odyn/odyn.db'


## Example Dropping Columns

In [ ]:
# Join example
df = db.from_query("""
   SELECT e.*, mc.* FROM experiments AS e
        JOIN group_experiments AS ge ON ge.exp_id   = e.exp_id
        JOIN groups            AS g  ON g.group_id  = ge.group_id
        JOIN method_calls      AS mc ON mc.group_id = g.group_id
        WHERE mc.method_name = 'Group.run_motion_correction'
              OR mc.method_name = 'Group.pick_mcor_parameters';
""")

# Remove unecessary columns
df = df.drop(
    columns=[
        "exp_id",
        "exp_type",
        "exp_start",
        "mouse_id",
        "frame_count",
        "frame_rate",
        "laser_power_920",
        "laser_power_1040",
        "loop_acq_interval_s",
        "call_log",
        "git_commit",
        "called_at",
        "method_call_id",
        "added_to_db_at",
        "call_flag"
    ]
)

df

## Selecting and Reordering Columns

### 2 method_calls

In [ ]:
# Join example
df1 = db.from_query("""
   SELECT e.*, mc.* FROM experiments AS e
        JOIN group_experiments AS ge ON ge.exp_id   = e.exp_id
        JOIN groups            AS g  ON g.group_id  = ge.group_id
        JOIN method_calls      AS mc ON mc.group_id = g.group_id
        WHERE mc.method_name = 'Group.run_motion_correction'
              OR mc.method_name = 'Group.pick_mcor_parameters';
""")

# Select columns to show and set their order
columns = [
 'method_name',
 'group_id',
 'exp_name',
 'height_um',
 'width_um',
 'height_px',
 'parameters',
 'call_output',
 ]
df1 = df1[columns]

# Add a column with new data
df1["um_per_pixel"] = df1.height_um / df1.height_px

# Select columns to show (including new column) and set their order
columns = [
 'method_name',
 'group_id',
 'exp_name',
 'height_um',
 'width_um',
 'um_per_pixel',
 'parameters',
 'call_output',
 ]
df1 = df1[columns]

df1

,method_name,group_id,exp_name,height_um,width_um,um_per_pixel,parameters,call_output
0,Group.pick_mcor_parameters,112,20260304_m357_e1,1200.0,1200.0,2.0,{},None
1,Group.pick_mcor_parameters,112,20260304_m357_e1,1200.0,1200.0,2.0,{},None
2,Group.pick_mcor_parameters,112,20260304_m357_e1,1200.0,1200.0,2.0,"{""frame_fraction"": 0.3}",None
3,Group.pick_mcor_parameters,175,20260706_m442_e1,600.0,1000.0,0.5,{},"{""strides_um"": [150.0, 125.0], ""overlap_um"": [..."
4,Group.run_motion_correction,175,20260706_m442_e1,600.0,1000.0,0.5,"{""is_test"": false, ""shifts_opencv"": false}",None


### 1 method_call + Expanding Parameters into Separate Columns

In [ ]:
# Join example
df2 = db.from_query("""
   SELECT e.*, mc.* FROM experiments AS e
        JOIN group_experiments AS ge ON ge.exp_id   = e.exp_id
        JOIN groups            AS g  ON g.group_id  = ge.group_id
        JOIN method_calls      AS mc ON mc.group_id = g.group_id
        WHERE mc.method_name = 'Group.run_motion_correction';
""")

# Select columns to show and set their order
columns = [
 'method_name',
 'group_id',
 'exp_name',
 'height_um',
 'width_um',
 'height_px',
 'parameters',
 ]
df2 = df2[columns]

# Add a column with new data
df2["um_per_pixel"] = df2.height_um / df2.height_px

# Select columns to show (including new column) and set their order
columns = [
 'method_name',
 'group_id',
 'exp_name',
 'height_um',
 'width_um',
 'um_per_pixel',
 'parameters',
 ]
df2 = df2[columns]

# Import stuff you need
import json
import pandas as pd

# Parse parameters string as a Python object
df2.parameters = df2.parameters.apply(json.loads)

# Expand parameters into multiple columns and concatenate along row axis (axis=1)
df2 = pd.concat([df2, pd.json_normalize(df2.parameters)], axis=1)

# Remove redundant parameters column
df2 = df2.drop(columns="parameters")

df2

,method_name,group_id,exp_name,height_um,width_um,um_per_pixel,is_test,shifts_opencv
0,Group.run_motion_correction,175,20260706_m442_e1,600.0,1000.0,0.5,False,False


## Filtering Examples

In [39]:
# filter example - remove empty call_output rows
df1[~df1.call_output.isna()]

,method_name,group_id,exp_name,height_um,width_um,um_per_pixel,parameters,call_output
3,Group.pick_mcor_parameters,175,20260706_m442_e1,600.0,1000.0,0.5,{},"{""strides_um"": [150.0, 125.0], ""overlap_um"": [..."


In [40]:
# filter example - show exp_name containing 442 and 2026
df1[(df1.exp_name.str.contains("442")) & (df1.exp_name.str.contains("2026"))]

,method_name,group_id,exp_name,height_um,width_um,um_per_pixel,parameters,call_output
3,Group.pick_mcor_parameters,175,20260706_m442_e1,600.0,1000.0,0.5,{},"{""strides_um"": [150.0, 125.0], ""overlap_um"": [..."
4,Group.run_motion_correction,175,20260706_m442_e1,600.0,1000.0,0.5,"{""is_test"": false, ""shifts_opencv"": false}",None


In [41]:
# How to find columns complementary to excluded columns
hidden_columns = {
        "exp_id",
        "exp_type",
        "exp_start",
        "mouse_id",
        "frame_count",
        "frame_rate",
        "laser_power_920",
        "laser_power_1040",
        "loop_acq_interval_s",
        "call_log",
        "git_commit",
        "called_at",
        "method_call_id",
        "added_to_db_at",
        "call_flag"
}

set(df1.columns).difference(hidden_columns)

{'call_output',
 'exp_name',
 'group_id',
 'height_um',
 'method_name',
 'parameters',
 'um_per_pixel',
 'width_um'}